# 05.2 · kv-transfer FAILURE analysis — single examples

Reconstructs instances that FAILED causally in a completed `kv_transfer_eval.py` run (kv-cluster or multi-predicate) and re-runs the same k+v transplant + oracle eval on them, one at a time, for qualitative debugging. Prompt rendering mirrors `05_kv_transfer.ipynb` — the **cached-from** (SRC) and **patched-to** (DST) token windows are marked, including the alignment **pads**. The intervention knobs default to the run_tag's values (`parse_run_tag`) but can be **overridden** below. No hparam search — just eyeballing *why* a case fails.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, glob, re
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4, 7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")
import torch, numpy as np, pandas as pd
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope (incl. align_kv_spans, headwise_kv_patch, ...)
import src.diff.data as D
import src.diff.eval as E
from contextlib import nullcontext

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")
mt = P.mt
GOAL_HEADS, UNION_HEADS = P.GOAL_HEADS, P.UNION_HEADS
READOUT_HEADS = GOAL_HEADS               # goalscope readout heads — always GOAL, kept separate from PATCH
from rich.panel import Panel; from rich.text import Text; from rich.markup import escape

2026-09-06 13:12:46 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Prompt renderer — cached-from / patched-to spans (+ pads)

`show_spans` (same as `05_kv_transfer.ipynb`): the full task text is `bright_yellow`; the exact **KV window** whose k+v are cached/overwritten is `black on dark_orange` (for `align='pad'` the whole span, pads included, is the window); the pivot is `salmon`.

In [2]:
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """task_span -> bright_yellow; kv_span -> black on dark_orange (cached-from / patched-to, incl. pads);
    pivot -> salmon. Priority pivot > kv > task so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos: s = PO + s + PC
        elif i in kv:   s = KO + s + KC
        elif i in task: s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Load failure cases, reconstruct instances, and filter

`rebuild()` reconstructs the kv / multi instance from a record. `filter_fails` narrows the pool by naming / datatype / lang / config / concept-pair; `show_fails` prints the breakdowns.

> **Effective naming:** multi configs were saved with raw naming `l/e/t/r` (the script's `--namings` default is the *string* `"letters"`, iterated into characters by `nargs="+"`). Only the literal `"letters"` renders positional `a/b/c` vars — every other value falls back to **natural** (descriptive) naming. So the completed multi runs are effectively ALL `natural`, which is why the failures cluster there. Filter on the **effective** scheme via `naming="natural"`.

In [3]:
MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"
MECHANISM = "kv_transfer"
_KVROOT = Path(P.env_utils.DEFAULT_RESULTS_DIR) / "diff" / MODEL_SHORT / MECHANISM
ROOTS = {"dual_comp": _KVROOT / "dual_comp", "multi_pred": _KVROOT / "multi_pred"}

def parse_run_tag(t):
    d = {"patch_heads": "goal", "ablate": "off", "ablation_strat": "none", "alpha": 1.0, "align": "pad"}
    for seg in t.split("__"):
        if seg.startswith("ph-"): d["patch_heads"] = seg[3:]
        elif seg.startswith("abl-"):
            rest = seg[4:]
            if rest in ("off", "none"): d["ablate"], d["ablation_strat"] = "off", "none"
            else:
                parts = rest.split("-")
                if parts[-1] in ("zero", "restrict"): d["ablation_strat"] = parts[-1]; parts = parts[:-1]
                d["ablate"] = "-".join(parts)
        elif re.fullmatch(r"a[0-9.]+", seg): d["alpha"] = float(seg[1:])
        elif seg in ("pad", "right"): d["align"] = seg
    return d

def _cluster_by_id(dt, cid):
    return next(c for c in D.KVCLUSTERS[dt] if c.id == cid)

def rebuild(r):
    dt = r["datatype"]
    if r.get("setup") == "dual_comp":
        cl = _cluster_by_id(dt, r["group"])
        return D.KVTransferInstance(cluster=cl, var_name=cl.var_names[0], presentation="description",
            src_lang=r["src_lang"], dst_lang=r["dst_lang"], src_task=r["expected_next"], dst_task=r["dst_next"],
            pivot="auto", meta={"datatype": dt, "cluster": cl.id, "dst_next": r["dst_next"]})
    bank = D.QTASKS[dt]; by = {p.id: p for p in bank.predicates}
    shared_ids = r["group"].split("shared:")[-1].split("-")
    return D.MultiPredicateInstance(datatype=dt, shared=tuple(by[i] for i in shared_ids),
        src_frontier=by[r["expected_next"]], dst_frontier=by[r["dst_next"]], var_name=bank.var_names[0],
        presentation="listed", src_lang=r["src_lang"], dst_lang=r["dst_lang"],
        naming=r["naming"] if r["naming"] in ("natural", "letters") else "natural",
        meta={"datatype": dt, "dst_next": r["dst_next"], "n_conditions": len(shared_ids) + 1})

def ablate_heads_for(knobs, patch_heads):
    if knobs["ablate"] in ("off", None):
        return []
    lo, hi = (int(x) for x in knobs["ablate"].split("-"))
    allh = [(l, h) for l in range(lo, hi + 1) for h in range(mt.n_heads)]
    return [h for h in allh if h not in set(patch_heads)]

def load_fails(setup, run_tag):
    out = []
    for rj in (ROOTS[setup] / run_tag).glob("*/results.json"):
        for r in json.loads(rj.read_text()):
            if r.get("causal") and not r["causal"]["triggered"]:
                r["_cfg"] = rj.parent.name; out.append(r)
    return out

def eff_naming(r):
    """EFFECTIVE variable-naming scheme actually rendered. NOTE: the batch script's --namings default
    is the *string* "letters", which (nargs="+") argparse iterates into characters -> multi configs got
    naming 'l','e','t','r','s'. Only the literal "letters" renders positional a/b/c vars; every other
    value (incl. those stray letters and 'natural') falls back to descriptive/natural naming in
    `var_of`. So filter on this, not the raw stored `naming`."""
    return "letters" if r.get("naming") == "letters" else "natural"

def filter_fails(fails, naming=None, datatype=None, lang=None, config=None, target=None,
                 frontier=None, ro_patched=None):
    """Narrow the failure pool. `naming` matches the EFFECTIVE scheme ('natural'|'letters', see
    eff_naming). `ro_patched` filters by the patched-readout label (e.g. 'dst' = the 'reverted'
    failures, 'src'/'both' = 'knew but didn't act')."""
    def ok(r):
        ro = (r.get("readout") or {}).get("patched")
        return (
            (naming is None or eff_naming(r) == naming)
            and (datatype is None or r["datatype"] == datatype)
            and (lang is None or f'{r["src_lang"]}->{r["dst_lang"]}' == lang)
            and (config is None or config in r["_cfg"])
            and (target is None or r["expected_next"] == target)
            and (frontier is None or r["dst_next"] == frontier)
            and (ro_patched is None or ro == ro_patched))
    return [r for r in fails if ok(r)]

def show_fails(fails, k=14):
    df = pd.DataFrame([dict(naming=r.get("naming"), eff=eff_naming(r), datatype=r["datatype"],
                            lang=f'{r["src_lang"]}->{r["dst_lang"]}',
                            ro_patched=(r.get("readout") or {}).get("patched")) for r in fails])
    print(f"{len(fails)} causal failures")
    if len(df):
        print("  by EFFECTIVE naming:", df.eff.value_counts().to_dict(),
              "  (raw stored naming:", df.naming.value_counts().to_dict(), ")")
        print("  by datatype       :", df.datatype.value_counts().to_dict())
        print("  by patched-readout:", df.ro_patched.value_counts().to_dict())
    for r in fails[:k]:
        ro = r.get("readout") or {}
        print(f"  [{r['_cfg']}] SRC {r['expected_next']!r} -> DST {r['dst_next']!r} "
              f"(naming={eff_naming(r)}) | causal={r['causal']['label']} · "
              f"readout {ro.get('source')}>{ro.get('dest')}>{ro.get('patched')}")
    return fails

SETUP, RUN_TAG = "multi_pred", "ph-goal__abl-33-43-zero__a2__pad"      # <- run to debug
all_fails = load_fails(SETUP, RUN_TAG)
show_fails(all_fails, k=4)

442 causal failures
  by EFFECTIVE naming: {'natural': 442}   (raw stored naming: {'e': 141, 'l': 139, 't': 121, 'r': 41} )
  by datatype       : {'list[str]': 442}
  by patched-readout: {'dst': 207, 'src': 124, 'neither': 103, 'both': 8}
  [multi__list_str__l__python-cpp] SRC 'count_ends_consonant' -> DST 'total_length' (naming=natural) | causal=neither · readout src>dst>dst
  [multi__list_str__l__python-cpp] SRC 'distinct_count' -> DST 'longest_word_length' (naming=natural) | causal=dst · readout src>dst>dst
  [multi__list_str__l__python-cpp] SRC 'count_capitalized' -> DST 'all_unique_lengths' (naming=natural) | causal=neither · readout src>dst>dst
  [multi__list_str__l__python-cpp] SRC 'count_all_lowercase' -> DST 'longest_word_length' (naming=natural) | causal=dst · readout src>dst>dst


[{'key': 'multi|list[str]|shared=count_short-total_uppercase_letters|src=count_ends_consonant|dst=total_length',
  'setup': 'multi',
  'datatype': 'list[str]',
  'group': 'shared:count_short-total_uppercase_letters',
  'naming': 'l',
  'presentation': 'listed',
  'src_lang': 'python',
  'dst_lang': 'cpp',
  'var_name': 'words',
  'expected_next': 'count_ends_consonant',
  'expected_next_desc': 'the number of words that end with a consonant',
  'dst_next': 'total_length',
  'dst_next_desc': 'the total number of characters across all words',
  'raw': {'src_prompt': '<|im_start|>system\nYou are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>\n<|im_start|>user\nWrite a Python program that implements a function `func` which, given a list of strings `words`, computes:\n(1) the number of words shorter than three characters\n(2) the total number of uppercase letters across all words\n(3) t

## Intervention knobs — from the run_tag, override as needed

`KNOBS = parse_run_tag(RUN_TAG)` gives the batch-run settings so the re-run reproduces the recorded verdict. Uncomment any line to override a single knob and probe a variation.

In [8]:
KNOBS = parse_run_tag(RUN_TAG)
# ── optional overrides (uncomment to deviate from the batch run) ──
KNOBS["alpha"] = 3.0                 # OV-boost of PATCH_HEADS on the patched gen/readout
# KNOBS["ablation_strat"] = "restrict" # "zero" | "restrict" | "none"
KNOBS["ablate"] = "23-43"            # ablate layer range, or "off"
# KNOBS["align"] = "pad"               # "pad" | "right"
# KNOBS["patch_heads"] = "union"       # "goal" | "union"

PATCH_HEADS = GOAL_HEADS if KNOBS["patch_heads"] == "goal" else UNION_HEADS
ABLATE_HEADS = ablate_heads_for(KNOBS, PATCH_HEADS)
print(f"SETUP={SETUP} · RUN_TAG={RUN_TAG}")
print(f"KNOBS={KNOBS}")
print(f"patch_heads={len(PATCH_HEADS)} · readout_heads={len(READOUT_HEADS)} · ablate_heads={len(ABLATE_HEADS)}")

SETUP=multi · RUN_TAG=ph-goal__abl-33-43-zero__a2__pad
KNOBS={'patch_heads': 'goal', 'ablate': '23-43', 'ablation_strat': 'zero', 'alpha': 3.0, 'align': 'pad'}
patch_heads=122 · readout_heads=122 · ablate_heads=799


## Select a failure and render the cached-from / patched-to spans

Filter the pool (e.g. `naming='natural'`), then pick by index or concept pair. The render uses `align_kv_spans(mode=KNOBS['align'])` — the same alignment the runner applies — so the orange window shows exactly which tokens (incl. pads) are cached from SRC and overwritten in DST.

In [9]:
# filter the pool, then pick one
pool = filter_fails(all_fails, naming="natural")      # <- naming="natural"|"letters" (effective) / datatype=... / ro_patched="dst"
print(f"{len(pool)} failures after filter (of {len(all_fails)})")

def pick(fs, i=0): return fs[i % len(fs)]
def pick_by_pair(fs, target, frontier):
    hits = [r for r in fs if r["expected_next"] == target and r["dst_next"] == frontier]
    return hits[0] if hits else None

rec = pick(pool, i=0)                                  # <- change i, or pick_by_pair(pool, "...", "...")
inst = rebuild(rec)
m = D.materialize(inst)
SRC_SPAN0 = token_span(m["src_prompt"], m["src_task_text"])   # pre-align located spans
DST_SPAN0 = token_span(m["dst_prompt"], m["dst_task_text"])
# align -> padded tokens + cached-from (SRC_POS) / patched-to (DST_POS) windows
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], SRC_SPAN0, DST_SPAN0, mode=KNOBS["align"],
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
piv_src, piv_dst = src_tok["input_ids"].shape[1] - 1, dst_tok["input_ids"].shape[1] - 1

print(f"[{rec['_cfg']}] naming={rec.get('naming')}  SRC.target={inst.expected_next!r}  <-  DST.frontier={inst.dst_next!r}")
print(f"  {inst.expected_next_desc!r}  vs  {inst.dst_next_desc!r}")
print(f"OLD batch verdict: causal={rec['causal']['label']} (FAILED) · readout_patched={rec.get('readout', {}).get('patched')}")
show_spans(src_tok, f"SRC · CACHED-FROM {tuple(SRC_SPAN)} (orange) · pivot@{piv_src} · align={KNOBS['align']}",
           task_span=SRC_SPAN, kv_span=(SRC_POS[0], SRC_POS[-1] + 1), piv=-1)
show_spans(dst_tok, f"DST · PATCHED-TO {tuple(DST_SPAN)} (orange) · pivot@{piv_dst} · align={KNOBS['align']}",
           task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), piv=-1)

442 failures after filter (of 442)
[multi__list_str__l__python-cpp] naming=l  SRC.target='count_ends_consonant'  <-  DST.frontier='total_length'
  'the number of words that end with a consonant'  vs  'the total number of characters across all words'
OLD batch verdict: causal=neither (FAILED) · readout_patched=dst


╭─ SRC · CACHED-FROM (63, 101) (orange) · pivot@157 · align=pad ──────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words shorter than three characters\n                                                         │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the number of words that end with a consonant\n                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     short_word_count = sum(1 for w in words if len(w) < 3)\n                                                    │
│     uppercase_total = sum(1 for w in words for c in w if c.isupper())\n                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · PATCHED-TO (64, 102) (orange) · pivot@212 · align=pad ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the number of words shorter than three characters\n                                                         │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of characters across all words\n                                                           │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     int short_word_count = 0;\n                                                                                 │
│     for(auto &short_word_count_w : words) if(short_word_count_w.size() < 3) short_word_count++;\n               │
│     int uppercase_total = 0;\n                                                                                  │
│     for(auto &uppercase_total_w : words) for(char uppercase_total_c : uppercase_total_w)                        │
│ if(std::isupper((unsigned char)uppercase_total_c)) uppercase_total++;\n                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Re-run the intervention — SRC / DST / patched generations + readouts

`run_kv_transfer_intervention` with `KNOBS` (it re-derives the same alignment internally). Grouped per run (red=source, green=destination, salmon=patched). Eyeball whether the patched DST's first computation jumps to the SRC target, reverts to its own frontier, or goes hybrid.

In [10]:
raw = P.run_kv_transfer_intervention(
    mt, inst, PATCH_HEADS, readout_heads=READOUT_HEADS,
    alpha=KNOBS["alpha"], align=KNOBS["align"],
    ablate_heads=ABLATE_HEADS, ablation_strat=KNOBS["ablation_strat"], max_new=96)

rich_print(f"[red1]source_run · clean SRC (target {inst.expected_next!r}):[/red1] \n{escape(raw['gen_src'])}")
rich_print(f"   ↳ goalscope · source_run: [italic]{escape(raw['ro_src'])}[/italic]")
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.dst_next!r}):[/sea_green3] \n{escape(raw['gen_dst'])}")
rich_print(f"   ↳ goalscope · destination_run: [italic]{escape(raw['ro_dst'])}[/italic]")
rich_print(f"[salmon1]patched_run · DST (expect SRC target {inst.expected_next!r}):[/salmon1] \n{escape(raw['gen_patched'])}")
rich_print(f"   ↳ goalscope · patched_run: [italic]{escape(raw['ro_patched'])}[/italic]")

ev = E.eval_kv_transfer(inst, patched_gen=raw["gen_patched"], readouts=(raw["ro_src"], raw["ro_dst"], raw["ro_patched"]))
rich_print(f"[bold]EVAL[/bold] · {ev.summary()}")
rich_print("  causal :", ev.causal)
rich_print("  readout:", ev.readout)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=157 '    ' => I'm checking if the last character of the string is not a vowel and is a letter.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=212 '    ' => I'm counting the total number of characters in the string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=212 '    ' => I'm counting characters in the string.

... (5 outputs left out)

↳ goalscope · patched_run: I'm counting characters in the string.

EVAL · causal: TRIGGERED (src) | readout: src→dst→dst

causal :
{
    'label': 'src',
    'why': 'The code checks whether each character is an uppercase consonant (isupper + not a vowel), which relates
to consonant detection as in sub-goal A, rather than simply counting total characters as in sub-goal B.',
    'raw': 'The code checks `std::isupper(c)` combined with a consonant filter (excluding vowels A, E, I, U, Y), 
meaning it\'s counting uppercase consonants — this corresponds to finding words/characters that end with a 
consonant or have consonant properties, aligning with sub-goal A (words ending with a consonant) in spirit, but 
actually it\'s iterating over characters checking if they\'re uppercase consonants.\n\nWait, let me re-read. The 
task says sub-goal A is "the number of words that end with a consonant" and sub-goal B is "the total number of 
characters across all words."\n\nThe code iterates over each character in each word and checks if it\'s an 
uppercase consonant. This is neither counting words that end with a consonant nor counting total characters. 
However, it\'s checking consonant properties (excluding vowels from uppercase letters), which is closer to sub-goal
A\'s consonant theme.\n\nBut strictly: sub-goal A = count words ending with consonant, sub-goal B = total character
count. The code counts uppercase consonants, which doesn\'t directly match either. But the consonant check 
(filtering out vowels) is more related to A than B.\n\n<verdict>A</verdict><why>The code checks whether each 
character is an uppercase consonant (isupper + not a vowel), which relates to consonant detection as in sub-goal A,
rather than simply counting total characters as in sub-goal B.</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'dst',
    'valid': True,
    'shifted': False,
    'details': {
        'source': {
            'label': 'src',
            'why': 'Checking if the last character is a letter that is not a vowel is precisely the logic needed to
determine if a word ends with a consonant, which is sub-goal A.',
            'raw': '<verdict>A</verdict><why>Checking if the last character is a letter that is not a vowel is 
precisely the logic needed to determine if a word ends with a consonant, which is sub-goal A.</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly states counting total characters in the string, which directly 
corresponds to sub-goal B (total number of characters across all words).',
            'raw': '<verdict>B</verdict><why>The phrase explicitly states counting total characters in the string, 
which directly corresponds to sub-goal B (total number of characters across all words).</why>'
        },
        'patched': {
            'label': 'dst',
            'why': 'Counting characters in the string directly corresponds to computing the total number of 
characters across all words, which is sub-goal B.',
            'raw': '<verdict>B</verdict>\n<why>Counting characters in the string directly corresponds to computing 
the total number of characters across all words, which is sub-goal B.</why>'
        }
    }
}

## Batch of (filtered) failures — quick re-run table

Re-run the first N failures from the current `pool` (respects the filter + KNOBS) and tabulate old vs new verdict + a patched-gen snippet, to spot patterns (revert-to-DST, hybrid, incoherent).

In [7]:
# N = 8
# rows = []
# for r in pool[:N]:
#     it = rebuild(r)
#     rw = P.run_kv_transfer_intervention(
#         mt, it, PATCH_HEADS, readout_heads=READOUT_HEADS, alpha=KNOBS["alpha"], align=KNOBS["align"],
#         ablate_heads=ABLATE_HEADS, ablation_strat=KNOBS["ablation_strat"], max_new=64)
#     e = E.eval_kv_transfer(it, patched_gen=rw["gen_patched"], readouts=(rw["ro_src"], rw["ro_dst"], rw["ro_patched"]))
#     rows.append(dict(
#         cfg=r["_cfg"].replace(f"{SETUP}__", "").replace("__python-python", "").replace("__python-cpp", "->cpp"),
#         naming=r.get("naming"), SRC=it.expected_next, DST=it.dst_next, old=r["causal"]["label"],
#         new=e.causal["label"], new_trig=e.causal["triggered"], ro_pat=e.readout["patched"],
#         gen=rw["gen_patched"][:50].replace(chr(10), " ")))
# res = pd.DataFrame(rows)
# print(f"re-ran {len(res)} failures ({SETUP} · {RUN_TAG} · KNOBS alpha={KNOBS['alpha']} strat={KNOBS['ablation_strat']}) "
#       f"· new triggers: {int(res.new_trig.sum())}")
# res